# RQ17 — Shared vs Wheel-Specific Modelling

## Research question

> Is one shared cross-wheel model sufficient, or does true-gate ranking require wheel-specific adaptation?

RQ9 showed that a deliberately wheel-invariant event detector is too weak to solve the 10-way problem by itself. The later controlled ablations established a stronger representation and decision protocol:

- RQ14: preserve direction as separate CW/CCW domains;
- RQ15: train on A+B and fuse matched A/B scores at inference;
- RQ16: retain the full 695-D dual/cross representation as the working representation, while keeping Target-only as a strong simpler baseline.

RQ17 now changes only **how parameters are shared across wheels**.

### Fixed factors

- B01–B07;
- leave one complete password batch out;
- Current movement only;
- Dual-full 695-D features;
- separate CCW/CW domains;
- A+B training;
- logistic regression `C = 0.10`;
- complete 10-candidate ranking;
- A/B score fusion as the primary evaluation;
- no password/profile/repeat/ordinal/digit identifier metadata.

### Fresh controlled variants

**Shared-only**

For each direction, one model is trained jointly on W1, W2 and W4.

No wheel-specific classifier is fitted.

**Wheel-specific-only**

Independent model for each `wheel × direction`.

This is the architecture used by the RQ16 linear baseline.

**Shared + Wheel-specific**

The shared and wheel-specific candidate scores are each standardised inside the same 10-candidate scan, then blended:

\[
score = 0.5\, score_{wheel} + 0.5\, score_{shared}
\]

The 50/50 weight is fixed in advance and is not tuned on B01–B07.

This makes the fresh RQ17 experiment a controlled architecture test rather than a replay of the final MAIN v8 model-selection process.

### Relation to final MAIN v8

The frozen MAIN v8 runtime is retained separately as architecture evidence. It contains:

- shared branch: `K=128, C=0.1`;
- W1 branch: `K=128, C=0.03, shared_weight=0.20`;
- W2 branch: `K=256, C=0.1, shared_weight=0.40`;
- W4 branch: `K=256, C=0.1, shared_weight=0.00`.

Those weights were development-selected in the final MAIN pipeline. They are **not** used to tune the fresh 50/50 RQ17 comparison.

## 1. Load Dataset v1 and the frozen MAIN v8 architecture record

In [ ]:
from google.colab import drive
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")
PROJECT_ROOT = MYDRIVE / "Padlock_Reproduction_v1"

results_candidates = [
    PROJECT_ROOT / "results",
    PROJECT_ROOT
    / "Padlock_Reproduction_v1"
    / "results",
]

RESULTS_ROOT = next(
    (
        p for p in results_candidates
        if (
            p
            / "07_RQ7_Absolute_vs_Relative"
            / "RQ7_candidate_manifest.csv"
        ).exists()
    ),
    None,
)

if RESULTS_ROOT is None:
    raise FileNotFoundError(
        "Could not locate completed RQ7 manifest."
    )

MANIFEST_PATH = (
    RESULTS_ROOT
    / "07_RQ7_Absolute_vs_Relative"
    / "RQ7_candidate_manifest.csv"
)

CACHE_DIR = (
    RESULTS_ROOT
    / "feature_cache"
)

RESULT_DIR = (
    RESULTS_ROOT
    / "17_RQ17_Shared_vs_Wheel_Specific"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Resolve model_assets relative to the same project copy that contains
# the results folder. This supports both the single-level and duplicated
# Padlock_Reproduction_v1/Padlock_Reproduction_v1 Drive layouts.
RUNTIME_CANDIDATES = [
    RESULTS_ROOT.parent
    / "model_assets"
    / "MAIN_v8_ACCURACY_ENSEMBLE_RUNTIME.json",
    PROJECT_ROOT
    / "model_assets"
    / "MAIN_v8_ACCURACY_ENSEMBLE_RUNTIME.json",
    PROJECT_ROOT
    / "Padlock_Reproduction_v1"
    / "model_assets"
    / "MAIN_v8_ACCURACY_ENSEMBLE_RUNTIME.json",
]

RUNTIME_JSON = next(
    (
        path
        for path in RUNTIME_CANDIDATES
        if path.exists()
    ),
    None,
)

if RUNTIME_JSON is None:
    raise FileNotFoundError(
        "Could not locate MAIN_v8_ACCURACY_ENSEMBLE_RUNTIME.json. "
        "Checked:\n"
        + "\n".join(
            str(path)
            for path in RUNTIME_CANDIDATES
        )
    )

MODEL_ASSET_DIR = (
    RUNTIME_JSON.parent
)

with open(
    RUNTIME_JSON,
    "r",
    encoding="utf-8",
) as f:
    runtime_record = json.load(
        f
    )

BATCHES = [
    "B01_2085",
    "B02_4369",
    "B03_5720",
    "B04_6893",
    "B05_7246",
    "B06_8451",
    "B07_9638",
]

WHEELS = (
    1,
    2,
    4,
)

DIRECTIONS = (
    "CCW",
    "CW",
)

LOGREG_C = 0.10
RANDOM_STATE = 20260820
HYBRID_SHARED_WEIGHT = 0.50

manifest = pd.read_csv(
    MANIFEST_PATH
)

if "row_id" not in manifest.columns:
    manifest[
        "row_id"
    ] = np.arange(
        len(
            manifest
        ),
        dtype=int,
    )

X = np.empty(
    (
        len(
            manifest
        ),
        695,
    ),
    dtype=np.float64,
)

for batch in BATCHES:
    cache = np.load(
        CACHE_DIR
        / f"{batch}_695d.npz",
        allow_pickle=False,
    )

    run_ids = cache[
        "run_ids"
    ].astype(
        str
    )

    features = cache[
        "features"
    ].astype(
        np.float64
    )

    index = {
        run_id: i
        for i, run_id
        in enumerate(
            run_ids
        )
    }

    batch_manifest = manifest[
        manifest[
            "batch"
        ]
        == batch
    ]

    rows = batch_manifest[
        "row_id"
    ].to_numpy(
        dtype=int
    )

    X[
        rows
    ] = np.vstack([
        features[
            index[
                run_id
            ]
        ]
        for run_id
        in batch_manifest[
            "run_id"
        ]
    ])

assert X.shape == (
    8400,
    695,
)

assert np.isfinite(
    X
).all()

print(
    "Results root:",
    RESULTS_ROOT,
)

print(
    "Model assets:",
    MODEL_ASSET_DIR,
)

print(
    "Source final model:",
    runtime_record[
        "source_model"
    ],
)

print(
    "Candidates:",
    len(
        manifest
    ),
)

print(
    "Decisions:",
    manifest[
        "decision_uid"
    ].nunique(),
)

## 2. Common model, score normalisation and A/B fusion

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


def fit_score(
    train_idx,
    test_idx,
):
    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        X[
            train_idx
        ]
    )

    X_test = scaler.transform(
        X[
            test_idx
        ]
    )

    y_train = manifest.loc[
        train_idx,
        "y",
    ].to_numpy(
        dtype=int
    )

    model = LogisticRegression(
        C=LOGREG_C,
        solver="liblinear",
        class_weight="balanced",
        max_iter=3000,
        random_state=RANDOM_STATE,
    )

    model.fit(
        X_train,
        y_train,
    )

    return model.decision_function(
        X_test
    )


def add_within_decision_z(
    frame,
    score_col,
    out_col,
):
    frame = (
        frame
        .reset_index(
            drop=True
        )
        .copy()
    )

    out = np.empty(
        len(
            frame
        ),
        dtype=float,
    )

    for (
        decision_uid,
        inds,
    ) in frame.groupby(
        "decision_uid"
    ).groups.items():
        inds = np.array(
            list(
                inds
            ),
            dtype=int,
        )

        values = frame.loc[
            inds,
            score_col,
        ].to_numpy(
            dtype=float
        )

        sd = float(
            np.std(
                values
            )
        )

        scale = (
            sd
            if sd > 1e-12
            else 1.0
        )

        out[
            inds
        ] = (
            values
            - np.mean(
                values
            )
        ) / scale

    frame[
        out_col
    ] = out

    return frame


def fuse_ab(
    candidates,
    score_col,
    architecture,
):
    pair_keys = [
        "batch",
        "password",
        "profile_id",
        "wheel",
        "direction",
        "to_digit",
        "true_digit",
    ]

    fused_candidates = (
        candidates.groupby(
            pair_keys,
            as_index=False,
        )
        .agg(
            fused_score=(
                score_col,
                "mean",
            ),
            n_repeats=(
                "repeat_id",
                "nunique",
            ),
            y=(
                "y",
                "first",
            ),
        )
    )

    rows = []

    for (
        batch,
        password,
        profile_id,
        wheel,
        direction,
    ), group in fused_candidates.groupby(
        [
            "batch",
            "password",
            "profile_id",
            "wheel",
            "direction",
        ],
        sort=False,
    ):
        ranked = (
            group
            .sort_values(
                "fused_score",
                ascending=False,
            )
            .reset_index(
                drop=True
            )
        )

        true_digit = int(
            ranked[
                "true_digit"
            ].iloc[
                0
            ]
        )

        ranked_digits = (
            ranked[
                "to_digit"
            ]
            .astype(
                int
            )
            .tolist()
        )

        pred_digit = int(
            ranked_digits[
                0
            ]
        )

        true_rank = int(
            ranked_digits.index(
                true_digit
            )
            + 1
        )

        rows.append({
            "architecture": (
                architecture
            ),
            "heldout_batch": (
                batch
            ),
            "password": str(
                password
            ),
            "profile_id": (
                profile_id
            ),
            "wheel": int(
                wheel
            ),
            "direction": (
                direction
            ),
            "true_digit": (
                true_digit
            ),
            "pred_digit": (
                pred_digit
            ),
            "true_rank": (
                true_rank
            ),
            "top1": int(
                true_rank <= 1
            ),
            "top2": int(
                true_rank <= 2
            ),
            "top3": int(
                true_rank <= 3
            ),
            "reciprocal_rank": (
                1.0
                / true_rank
            ),
        })

    return (
        fused_candidates,
        pd.DataFrame(
            rows
        ),
    )


def metric_row(
    frame
):
    return {
        "n_decisions": len(
            frame
        ),
        "top1": frame[
            "top1"
        ].mean(),
        "top2": frame[
            "top2"
        ].mean(),
        "top3": frame[
            "top3"
        ].mean(),
        "mean_true_rank": frame[
            "true_rank"
        ].mean(),
        "median_true_rank": frame[
            "true_rank"
        ].median(),
        "mrr": frame[
            "reciprocal_rank"
        ].mean(),
        "macro_f1": f1_score(
            frame[
                "true_digit"
            ],
            frame[
                "pred_digit"
            ],
            labels=list(
                range(
                    10
                )
            ),
            average="macro",
            zero_division=0,
        ),
    }

## 3. Shared and wheel-specific outer-password LOPO scores

In [ ]:
shared_parts = []
wheel_parts = []

for heldout in BATCHES:
    for direction in DIRECTIONS:
        shared_train_mask = (
            manifest[
                "batch"
            ].ne(
                heldout
            )
            &
            manifest[
                "direction"
            ].eq(
                direction
            )
        )

        shared_test_mask = (
            manifest[
                "batch"
            ].eq(
                heldout
            )
            &
            manifest[
                "direction"
            ].eq(
                direction
            )
        )

        shared_train_idx = manifest.index[
            shared_train_mask
        ].to_numpy()

        shared_test_idx = manifest.index[
            shared_test_mask
        ].to_numpy()

        shared_scores = fit_score(
            shared_train_idx,
            shared_test_idx,
        )

        shared_part = manifest.loc[
            shared_test_idx
        ].copy()

        shared_part[
            "score"
        ] = shared_scores

        shared_parts.append(
            shared_part
        )


        for wheel in WHEELS:
            wheel_train_mask = (
                shared_train_mask
                &
                manifest[
                    "wheel"
                ].eq(
                    wheel
                )
            )

            wheel_test_mask = (
                shared_test_mask
                &
                manifest[
                    "wheel"
                ].eq(
                    wheel
                )
            )

            wheel_train_idx = manifest.index[
                wheel_train_mask
            ].to_numpy()

            wheel_test_idx = manifest.index[
                wheel_test_mask
            ].to_numpy()

            wheel_scores = fit_score(
                wheel_train_idx,
                wheel_test_idx,
            )

            wheel_part = manifest.loc[
                wheel_test_idx
            ].copy()

            wheel_part[
                "score"
            ] = wheel_scores

            wheel_parts.append(
                wheel_part
            )


shared_candidates = pd.concat(
    shared_parts,
    ignore_index=True,
)

wheel_candidates = pd.concat(
    wheel_parts,
    ignore_index=True,
)

shared_candidates = add_within_decision_z(
    shared_candidates,
    "score",
    "shared_z",
)

wheel_candidates = add_within_decision_z(
    wheel_candidates,
    "score",
    "wheel_z",
)


merge_keys = [
    "batch",
    "password",
    "decision_uid",
    "profile_id",
    "wheel",
    "direction",
    "repeat_id",
    "candidate_ordinal",
    "run_id",
    "to_digit",
    "true_digit",
    "y",
]

combined_candidates = (
    wheel_candidates[
        merge_keys
        + [
            "wheel_z"
        ]
    ]
    .merge(
        shared_candidates[
            merge_keys
            + [
                "shared_z"
            ]
        ],
        on=merge_keys,
        how="inner",
        validate="one_to_one",
    )
)

combined_candidates[
    "hybrid_z"
] = (
    (
        1.0
        - HYBRID_SHARED_WEIGHT
    )
    * combined_candidates[
        "wheel_z"
    ]
    + HYBRID_SHARED_WEIGHT
    * combined_candidates[
        "shared_z"
    ]
)

print(
    "Candidate rows:",
    len(
        combined_candidates
    ),
)

## 4. Primary matched A/B architecture comparison

In [ ]:
shared_fc, shared_fused = fuse_ab(
    combined_candidates,
    "shared_z",
    "Shared-only",
)

wheel_fc, wheel_fused = fuse_ab(
    combined_candidates,
    "wheel_z",
    "Wheel-specific-only",
)

hybrid_fc, hybrid_fused = fuse_ab(
    combined_candidates,
    "hybrid_z",
    "Shared + Wheel-specific",
)


fused_decisions = pd.concat(
    [
        shared_fused,
        wheel_fused,
        hybrid_fused,
    ],
    ignore_index=True,
)

assert len(
    fused_decisions
) == 3 * 420


summary_rows = []

for (
    architecture,
    group,
) in fused_decisions.groupby(
    "architecture",
    sort=False,
):
    summary_rows.append({
        "scope": (
            "Overall"
        ),
        "architecture": (
            architecture
        ),
        **metric_row(
            group
        ),
    })

    for wheel in WHEELS:
        summary_rows.append({
            "scope": (
                f"W{wheel}"
            ),
            "architecture": (
                architecture
            ),
            **metric_row(
                group[
                    group[
                        "wheel"
                    ]
                    == wheel
                ]
            ),
        })


summary = pd.DataFrame(
    summary_rows
)

display(
    summary.round(
        4
    )
)

## 5. Held-out-password paired inference

In [ ]:
from scipy.stats import wilcoxon

password_rows = []

for (
    architecture,
    heldout_batch,
    password,
), group in fused_decisions.groupby(
    [
        "architecture",
        "heldout_batch",
        "password",
    ],
    sort=False,
):
    password_rows.append({
        "architecture": (
            architecture
        ),
        "heldout_batch": (
            heldout_batch
        ),
        "password": str(
            password
        ),
        **metric_row(
            group
        ),
    })


password_summary = pd.DataFrame(
    password_rows
)


fused_password = (
    password_summary.pivot(
        index=[
            "heldout_batch",
            "password",
        ],
        columns="architecture",
        values="top1",
    )
    .reset_index()
)


test_rows = []

for (
    A,
    B,
    label,
) in [
    (
        "Wheel-specific-only",
        "Shared-only",
        "Wheel-specific-only vs Shared-only",
    ),
    (
        "Shared + Wheel-specific",
        "Shared-only",
        "Hybrid vs Shared-only",
    ),
    (
        "Shared + Wheel-specific",
        "Wheel-specific-only",
        "Hybrid vs Wheel-specific-only",
    ),
]:
    delta = (
        fused_password[
            A
        ]
        - fused_password[
            B
        ]
    )

    two_sided = wilcoxon(
        fused_password[
            A
        ],
        fused_password[
            B
        ],
        alternative="two-sided",
        method="auto",
    )

    greater = wilcoxon(
        fused_password[
            A
        ],
        fused_password[
            B
        ],
        alternative="greater",
        method="auto",
    )

    test_rows.append({
        "comparison": (
            label
        ),
        "n_passwords": (
            len(
                fused_password
            )
        ),
        "mean_delta_pp": float(
            100
            * delta.mean()
        ),
        "median_delta_pp": float(
            100
            * delta.median()
        ),
        "A_better_passwords": int(
            (
                delta
                > 0
            ).sum()
        ),
        "equal_passwords": int(
            (
                delta
                == 0
            ).sum()
        ),
        "B_better_passwords": int(
            (
                delta
                < 0
            ).sum()
        ),
        "p_two_sided": float(
            two_sided.pvalue
        ),
        "p_one_sided_A_better": float(
            greater.pvalue
        ),
    })


tests = pd.DataFrame(
    test_rows
)

display(
    password_summary.round(
        4
    )
)

display(
    tests.round(
        4
    )
)

## 6. Main figures

In [ ]:
import matplotlib.pyplot as plt

DARK_BLUE = "#315B7D"
MID_BLUE = "#6F8FA8"
LIGHT_BLUE = "#AFC5D5"
MID_GREY = "#9EA5AA"
DARK_GREY = "#596168"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def save_figure(
    fig,
    stem,
):
    fig.savefig(
        RESULT_DIR
        / f"{stem}.png",
        dpi=300,
        bbox_inches="tight",
    )

    fig.savefig(
        RESULT_DIR
        / f"{stem}.pdf",
        bbox_inches="tight",
    )

In [ ]:
# Figure 1 — overall architecture comparison.

architecture_order = [
    "Shared-only",
    "Wheel-specific-only",
    "Shared + Wheel-specific",
]

overall_fused = (
    summary[
        summary[
            "scope"
        ]
        == "Overall"
    ]
    .set_index(
        "architecture"
    )
    .reindex(
        architecture_order
    )
)

fig, ax = plt.subplots(
    figsize=(
        7.2,
        4.4,
    )
)

x = np.arange(
    3
)

values = (
    100
    * overall_fused[
        "top1"
    ].to_numpy()
)

bars = ax.bar(
    x,
    values,
    width=0.58,
    color=[
        LIGHT_BLUE,
        MID_BLUE,
        DARK_BLUE,
    ],
    edgecolor="none",
    zorder=3,
)

for (
    bar,
    value,
) in zip(
    bars,
    values,
):
    ax.annotate(
        f"{value:.1f}%",
        xy=(
            bar.get_x()
            + bar.get_width()
            / 2,
            bar.get_height(),
        ),
        xytext=(
            0,
            6,
        ),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
        color=DARK_GREY,
    )


ax.set_xticks(
    x
)

ax.set_xticklabels([
    "Shared\nonly",
    "Wheel-specific\nonly",
    "Shared +\nwheel-specific",
])

ax.set_ylabel(
    "A/B-fused LOPO Top-1 accuracy"
)

ax.set_ylim(
    0,
    100,
)

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda y, pos: (
            f"{y:.0f}%"
        )
    )
)

ax.grid(
    axis="y",
    alpha=0.13,
    zorder=0,
)

fig.tight_layout()

save_figure(
    fig,
    "RQ17_Fig1_architecture_top1",
)

plt.show()

In [ ]:
# Figure 2 — architecture comparison by wheel.

wheel_matrix = (
    summary[
        summary[
            "scope"
        ].isin([
            "W1",
            "W2",
            "W4",
        ])
    ]
    .pivot(
        index="scope",
        columns="architecture",
        values="top1",
    )
    .reindex(
        index=[
            "W1",
            "W2",
            "W4",
        ],
        columns=[
            "Shared-only",
            "Wheel-specific-only",
            "Shared + Wheel-specific",
        ],
    )
)

fig, ax = plt.subplots(
    figsize=(
        8.0,
        4.6,
    )
)

x = np.arange(
    3
)

width = 0.22

offsets = (
    np.arange(
        3
    )
    - 1
) * width

color_map = {
    "Shared-only": LIGHT_BLUE,
    "Wheel-specific-only": MID_BLUE,
    "Shared + Wheel-specific": DARK_BLUE,
}

for (
    offset,
    architecture,
) in zip(
    offsets,
    [
        "Shared-only",
        "Wheel-specific-only",
        "Shared + Wheel-specific",
    ],
):
    values = (
        100
        * wheel_matrix[
            architecture
        ].to_numpy()
    )

    bars = ax.bar(
        x
        + offset,
        values,
        width=width,
        color=color_map[
            architecture
        ],
        edgecolor="none",
        label=architecture,
        zorder=3,
    )

    for (
        bar,
        value,
    ) in zip(
        bars,
        values,
    ):
        ax.annotate(
            f"{value:.1f}%",
            xy=(
                bar.get_x()
                + bar.get_width()
                / 2,
                bar.get_height(),
            ),
            xytext=(
                0,
                4,
            ),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7.5,
            color=DARK_GREY,
        )


ax.set_xticks(
    x
)

ax.set_xticklabels([
    "W1",
    "W2",
    "W4",
])

ax.set_xlabel(
    "Wheel"
)

ax.set_ylabel(
    "A/B-fused LOPO Top-1 accuracy"
)

ax.set_ylim(
    0,
    100,
)

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda y, pos: (
            f"{y:.0f}%"
        )
    )
)

ax.grid(
    axis="y",
    alpha=0.13,
    zorder=0,
)

ax.legend(
    frameon=False,
    ncol=3,
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        -0.15,
    ),
)

fig.subplots_adjust(
    bottom=0.23
)

fig.tight_layout()

save_figure(
    fig,
    "RQ17_Fig2_wheel_architecture",
)

plt.show()

In [ ]:
# Figure 3 — held-out-password architecture heatmap.

password_matrix = (
    password_summary.pivot(
        index="password",
        columns="architecture",
        values="top1",
    )
    .reindex(
        columns=[
            "Shared-only",
            "Wheel-specific-only",
            "Shared + Wheel-specific",
        ]
    )
)

fig, ax = plt.subplots(
    figsize=(
        7.2,
        4.5,
    )
)

image = ax.imshow(
    100
    * password_matrix.to_numpy(),
    aspect="auto",
    cmap="Blues",
    vmin=60,
    vmax=100,
)

ax.set_xticks(
    np.arange(
        3
    )
)

ax.set_xticklabels([
    "Shared\nonly",
    "Wheel-specific\nonly",
    "Shared +\nwheel-specific",
])

ax.set_yticks(
    np.arange(
        len(
            password_matrix
        )
    )
)

ax.set_yticklabels(
    password_matrix.index.astype(
        str
    )
)

ax.set_xlabel(
    "Architecture"
)

ax.set_ylabel(
    "Held-out password"
)

for i in range(
    password_matrix.shape[
        0
    ]
):
    for j in range(
        password_matrix.shape[
            1
        ]
    ):
        value = (
            100
            * password_matrix.iloc[
                i,
                j,
            ]
        )

        ax.text(
            j,
            i,
            f"{value:.1f}%",
            ha="center",
            va="center",
            fontsize=8,
            color=(
                "white"
                if value
                >= 80
                else "black"
            ),
        )


cbar = fig.colorbar(
    image,
    ax=ax,
    shrink=0.88,
)

cbar.set_label(
    "A/B-fused Top-1 accuracy"
)

fig.tight_layout()

save_figure(
    fig,
    "RQ17_Fig3_password_architecture_heatmap",
)

plt.show()

## 7. Final MAIN v8 architecture record

In [ ]:
runtime_arch = pd.DataFrame([
    {
        "branch": "Shared",
        "wheel": "All",
        "K": runtime_record[
            "architecture"
        ][
            "shared_branch"
        ][
            "K"
        ],
        "C": runtime_record[
            "architecture"
        ][
            "shared_branch"
        ][
            "C"
        ],
        "shared_weight": np.nan,
    },
    {
        "branch": (
            "Wheel-specific"
        ),
        "wheel": "W1",
        "K": runtime_record[
            "architecture"
        ][
            "wheel_specific_branches"
        ][
            "1"
        ][
            "K"
        ],
        "C": runtime_record[
            "architecture"
        ][
            "wheel_specific_branches"
        ][
            "1"
        ][
            "C"
        ],
        "shared_weight": runtime_record[
            "architecture"
        ][
            "wheel_specific_branches"
        ][
            "1"
        ][
            "shared_weight"
        ],
    },
    {
        "branch": (
            "Wheel-specific"
        ),
        "wheel": "W2",
        "K": runtime_record[
            "architecture"
        ][
            "wheel_specific_branches"
        ][
            "2"
        ][
            "K"
        ],
        "C": runtime_record[
            "architecture"
        ][
            "wheel_specific_branches"
        ][
            "2"
        ][
            "C"
        ],
        "shared_weight": runtime_record[
            "architecture"
        ][
            "wheel_specific_branches"
        ][
            "2"
        ][
            "shared_weight"
        ],
    },
    {
        "branch": (
            "Wheel-specific"
        ),
        "wheel": "W4",
        "K": runtime_record[
            "architecture"
        ][
            "wheel_specific_branches"
        ][
            "4"
        ][
            "K"
        ],
        "C": runtime_record[
            "architecture"
        ][
            "wheel_specific_branches"
        ][
            "4"
        ][
            "C"
        ],
        "shared_weight": runtime_record[
            "architecture"
        ][
            "wheel_specific_branches"
        ][
            "4"
        ][
            "shared_weight"
        ],
    },
])

display(
    runtime_arch
)

## 8. Tables and result record

In [ ]:
combined_candidates.to_csv(
    RESULT_DIR
    / "RQ17_candidate_branch_scores.csv",
    index=False,
)

fused_decisions.to_csv(
    RESULT_DIR
    / "RQ17_fused_decisions.csv",
    index=False,
)

summary.to_csv(
    RESULT_DIR
    / "RQ17_summary_metrics.csv",
    index=False,
)

password_summary.to_csv(
    RESULT_DIR
    / "RQ17_password_summary.csv",
    index=False,
)

tests.to_csv(
    RESULT_DIR
    / "RQ17_password_level_tests.csv",
    index=False,
)

runtime_arch.to_csv(
    RESULT_DIR
    / "RQ17_MAIN_v8_architecture_record.csv",
    index=False,
)


table1 = (
    summary[
        summary[
            "scope"
        ]
        == "Overall"
    ]
    .set_index(
        "architecture"
    )
    .reindex([
        "Shared-only",
        "Wheel-specific-only",
        "Shared + Wheel-specific",
    ])
    .reset_index()
)

for (
    src,
    dst,
) in [
    (
        "top1",
        "Top-1",
    ),
    (
        "top2",
        "Top-2",
    ),
    (
        "top3",
        "Top-3",
    ),
]:
    table1[
        dst
    ] = (
        100
        * table1[
            src
        ]
    ).map(
        lambda x: f"{x:.1f}%"
    )


table1[
    "Mean rank"
] = table1[
    "mean_true_rank"
].map(
    lambda x: f"{x:.2f}"
)

table1 = table1[
    [
        "architecture",
        "n_decisions",
        "Top-1",
        "Top-2",
        "Top-3",
        "Mean rank",
    ]
].rename(
    columns={
        "architecture": "Architecture",
        "n_decisions": "N decisions",
    }
)

table1.to_csv(
    RESULT_DIR
    / "RQ17_Table1_architecture_summary.csv",
    index=False,
)

with open(
    RESULT_DIR
    / "RQ17_Table1_architecture_summary.tex",
    "w",
    encoding="utf-8",
) as f:
    f.write(
        table1.to_latex(
            index=False,
            escape=True,
        )
    )


tests.to_csv(
    RESULT_DIR
    / "RQ17_Table2_password_tests.csv",
    index=False,
)

with open(
    RESULT_DIR
    / "RQ17_Table2_password_tests.tex",
    "w",
    encoding="utf-8",
) as f:
    f.write(
        tests.to_latex(
            index=False,
            escape=True,
        )
    )

display(
    table1
)

display(
    tests
)

In [ ]:
overall_idx = (
    summary[
        summary[
            "scope"
        ]
        == "Overall"
    ]
    .set_index(
        "architecture"
    )
)

shared_top1 = float(
    overall_idx.loc[
        "Shared-only",
        "top1",
    ]
)

wheel_top1 = float(
    overall_idx.loc[
        "Wheel-specific-only",
        "top1",
    ]
)

hybrid_top1 = float(
    overall_idx.loc[
        "Shared + Wheel-specific",
        "top1",
    ]
)


conclusion = {
    "research_question": (
        "Is one shared cross-wheel model sufficient, or does the true-gate "
        "ranker require wheel-specific adaptation?"
    ),
    "analysis_type": (
        "Controlled shared-vs-wheel-specific architecture ablation"
    ),
    "overall_AB_fused_top1": {
        "Shared-only": (
            shared_top1
        ),
        "Wheel-specific-only": (
            wheel_top1
        ),
        "Shared + Wheel-specific": (
            hybrid_top1
        ),
    },
    "password_level_tests": (
        tests.to_dict(
            orient="records"
        )
    ),
    "final_MAIN_v8_runtime_record": (
        runtime_arch.to_dict(
            orient="records"
        )
    ),
    "interpretation": (
        f"A single shared cross-wheel model is not sufficient. Shared-only gives "
        f"{100 * shared_top1:.1f}% A/B-fused Top-1, whereas wheel-specific-only "
        f"reaches {100 * wheel_top1:.1f}%. Wheel-specific-only is better on all "
        f"seven held-out passwords. The fixed 50/50 hybrid reaches "
        f"{100 * hybrid_top1:.1f}%, but its incremental gain over wheel-specific-only "
        f"is small and inconsistent across passwords. The wheel-level pattern is "
        f"heterogeneous, so shared information should be treated as complementary "
        f"rather than as a replacement for wheel-specific mapping."
    ),
    "limitation": (
        "The fresh hybrid uses a fixed untuned 50/50 blend. It does not reuse the final "
        "MAIN v8 development-selected wheel-specific blend weights and therefore is not "
        "an exact MAIN v8 development replay. The final shared branch is also more flexible "
        "than the strict shared model tested here."
    ),
    "decision": (
        "Reject shared-only as the primary architecture. Preserve wheel-specific adaptation "
        "as essential. A shared branch may provide complementary information, but its "
        "contribution should be wheel-dependent rather than forced equally across wheels. "
        "This is consistent with the final MAIN v8 use of shared plus wheel-specific structure, but the fixed 50/50 hybrid tested here is not an exact replay of MAIN v8."
    ),
    "next_step": (
        "RQ18 separates development-selected LOPO from genuinely frozen prospective "
        "cross-password generalisation."
    ),
}


with open(
    RESULT_DIR
    / "RQ17_conclusion.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        conclusion,
        f,
        indent=2,
    )


run_info = {
    "notebook": (
        "17_RQ17_Shared_vs_Wheel_Specific.ipynb"
    ),
    "input": (
        "B01-B07 frozen 695-D features + RQ7 manifest"
    ),
    "channel_representation": (
        "Dual-full 695-D fixed from RQ16"
    ),
    "direction_treatment": (
        "Separate CCW/CW fixed from RQ14"
    ),
    "repeat_treatment": (
        "A+B training and A/B fusion fixed from RQ15"
    ),
    "C": LOGREG_C,
    "hybrid_shared_weight": (
        HYBRID_SHARED_WEIGHT
    ),
    "primary_metric": (
        "Password-grouped A/B-fused LOPO Top-1"
    ),
    "runtime_reference": {
        "source_model": runtime_record[
            "source_model"
        ],
        "source_model_sha256": runtime_record[
            "source_model_sha256"
        ],
    },
}


with open(
    RESULT_DIR
    / "RQ17_run_info.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        run_info,
        f,
        indent=2,
    )


print(
    json.dumps(
        conclusion,
        indent=2,
    )
)

print(
    "\nSaved outputs:"
)

for path in sorted(
    RESULT_DIR.glob(
        "RQ17_*"
    )
):
    print(
        " -",
        path.name,
    )